# **Build Visual Embeddings and Upload Them to Hugging Face**

This notebook runs the complete pipeline in order: clone the `khang1108/MLeCDanBGold` repository, create the `aic/` virtual environment, download the dataset from Hugging Face, prepare `frames.parquet` when necessary, automatically select `cuda` or `cpu`, run `scripts/build_embeddings.py`, validate the outputs, and upload `artifacts/embeddings` to Hugging Face.

> Before starting, select a GPU runtime when one is available. In Google Colab, choose **Runtime → Change runtime type → T4 GPU**. Run the cells from top to bottom; there is no need to change the working directory manually.

# **1. Bootstrap Imports**

This cell imports only Python standard-library modules required to clone the repository and manage subprocesses. Project dependencies are installed in the next section, while all user-editable settings remain in a separate configuration cell.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# **2. Clone the Repository**

The repository is cloned to `/content/MLeCDanBGold` on Google Colab or beneath the current working directory on a regular VM/Jupyter environment. If the repository already exists, the notebook reuses it without automatically pulling or resetting, which prevents accidental overwrites of local changes.

In [ ]:
REPO_URL = "https://github.com/khang1108/MLeCDanBGold.git"
CURRENT_DIR = Path.cwd().resolve()
WORKSPACE = Path("/content") if Path("/content").is_dir() else CURRENT_DIR
PROJECT_DIR = (
    CURRENT_DIR
    if (CURRENT_DIR / ".git").is_dir() and (CURRENT_DIR / "pyproject.toml").is_file()
    else WORKSPACE / "MLeCDanBGold"
)

if not (PROJECT_DIR / ".git").is_dir():
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
else:
    print(f"Reuse existing repository: {PROJECT_DIR}")

os.chdir(PROJECT_DIR)
print(f"Project root: {Path.cwd()}")

# **3. Install Dependencies**

On a regular VM, the project uses the repository-standard `aic/` virtual environment. Google Colab and Kaggle already run inside managed Python environments where `venv`/`ensurepip` may be unavailable, so the notebook installs the project into the active kernel environment there. The selected interpreter is stored in `PYTHON` and used consistently by every project script.

In [ ]:
%pip install -q "huggingface_hub>=0.25" "pyyaml>=6,<7" "pandas>=2,<3" "pyarrow>=14,<22"

In [ ]:
IS_HOSTED_NOTEBOOK = (
    Path("/content").is_dir()
    or Path("/kaggle/working").is_dir()
    or "COLAB_RELEASE_TAG" in os.environ
    or "KAGGLE_KERNEL_RUN_TYPE" in os.environ
)

if IS_HOSTED_NOTEBOOK:
    PYTHON = Path(sys.executable)
    print("Hosted notebook detected; using the active kernel environment.")
else:
    VENV_DIR = PROJECT_DIR / "aic"
    if not (VENV_DIR / "bin" / "python").is_file():
        subprocess.run([sys.executable, "-m", "venv", str(VENV_DIR)], check=True)
    PYTHON = VENV_DIR / "bin" / "python"

subprocess.run(
    [str(PYTHON), "-m", "pip", "install", "-e", ".[embedding,dev]"],
    cwd=PROJECT_DIR,
    check=True,
)
subprocess.run(
    [
        str(PYTHON),
        "-c",
        "import faiss, hcmai, pandas, pyarrow, torch, transformers, yaml; print('Embedding environment: OK')",
    ],
    cwd=PROJECT_DIR,
    check=True,
)
print(f"Project Python: {PYTHON}")

# **4. Import Notebook Pipeline Dependencies**

These imports support Hugging Face downloads/uploads, runtime YAML generation, and `frames.parquet` validation. This cell is intentionally separate from the configuration cell so settings can be changed and rerun without repeating imports.

In [ ]:
import pandas as pd
import yaml
from huggingface_hub import HfApi, snapshot_download

# **5. Configure the Run**

Set `HF_TOKEN` to your Hugging Face token. Read access is sufficient to download a public dataset or a private dataset you can access; write access is required for uploading. By default, results are uploaded to `huylhn1810/aic_2025` under `artifacts/embeddings`. Change `HF_UPLOAD_REPO` if you want to use your own dataset repository.

`GPU_BATCH_SIZE` and `CPU_BATCH_SIZE` are the first values to tune when VRAM or RAM is limited. Reduce the batch size if CUDA runs out of memory or the machine starts using substantial swap space.

In [ ]:
# Hugging Face — paste the token below and never commit a notebook containing it.
HF_TOKEN = ""
HF_DATASET_REPO = "huylhn1810/aic_2025"
HF_UPLOAD_REPO = "huylhn1810/aic_2025"
HF_UPLOAD_PATH = "artifacts/embeddings"

# Local paths.
DATASET_ROOT = PROJECT_DIR / "data"
FRAMES_PATH = DATASET_ROOT / "metadata" / "frames.parquet"
OUTPUT_ROOT = PROJECT_DIR / "artifacts"
EMBEDDINGS_DIR = OUTPUT_ROOT / "embeddings"
BASE_CONFIG = PROJECT_DIR / "configs" / "baseline.yaml"
RUNTIME_CONFIG = OUTPUT_ROOT / "configs" / "notebook_embedding.yaml"

# Embedding settings.
MODEL_NAME = "google/siglip2-base-patch16-224"
GPU_BATCH_SIZE = 32
CPU_BATCH_SIZE = 4
DATASET_VERSION = "aic_2025_hf"

# Set allow_patterns, for example ["metadata/**", "features/**", "keyframes/**"],
# to limit downloaded files. None downloads the complete dataset snapshot.
HF_ALLOW_PATTERNS = None

# **6. Detect the Device and Generate a Runtime Config**

The notebook checks the PyTorch installation inside `aic/`. It selects `cuda` when `torch.cuda.is_available()` returns `True` and otherwise falls back to `cpu`. It then copies `configs/baseline.yaml` into a runtime config and updates only the fields relevant to this run; the original baseline config is not modified.

In [ ]:
# Reconstruct the interpreter path instead of relying on state from section 3.
IS_HOSTED_NOTEBOOK = (
    Path("/content").is_dir()
    or Path("/kaggle/working").is_dir()
    or "COLAB_RELEASE_TAG" in os.environ
    or "KAGGLE_KERNEL_RUN_TYPE" in os.environ
)
PYTHON = Path(sys.executable) if IS_HOSTED_NOTEBOOK else PROJECT_DIR / "aic" / "bin" / "python"
if not PYTHON.is_file():
    raise FileNotFoundError(
        f"Project environment not found at {PYTHON}. Run section 3 first."
    )

device_probe = subprocess.run(
    [str(PYTHON), "-c", "import torch; print('cuda' if torch.cuda.is_available() else 'cpu')"],
    capture_output=True,
    text=True,
    check=True,
)
DEVICE = device_probe.stdout.strip()
BATCH_SIZE = GPU_BATCH_SIZE if DEVICE == "cuda" else CPU_BATCH_SIZE

with BASE_CONFIG.open(encoding="utf-8") as config_file:
    runtime_config = yaml.safe_load(config_file)

runtime_config["dataset"].update({
    "version": DATASET_VERSION,
    "root": str(DATASET_ROOT),
    "frames_path": str(FRAMES_PATH),
})
runtime_config["models"]["embedding"].update({
    "name": MODEL_NAME,
    "device": DEVICE,
    "batch_size": BATCH_SIZE,
})
runtime_config["index"]["path"] = str(OUTPUT_ROOT / "indexes" / "visual.index")

RUNTIME_CONFIG.parent.mkdir(parents=True, exist_ok=True)
with RUNTIME_CONFIG.open("w", encoding="utf-8") as config_file:
    yaml.safe_dump(runtime_config, config_file, sort_keys=False)

print(f"Device     : {DEVICE}")
print(f"Batch size : {BATCH_SIZE}")
print(f"Config     : {RUNTIME_CONFIG}")

# **7. Download the Dataset from Hugging Face**

`snapshot_download` downloads a versioned snapshot and reuses cached files on subsequent runs. The repository layout is preserved beneath `data/`. The pipeline expects `data/keyframes` and `data/features/map-keyframes`, and it reuses `data/metadata/frames.parquet` when that file is already present.

> For a public dataset repository, the token may remain empty during download. The notebook never prints the token.

In [ ]:
DATASET_ROOT.mkdir(parents=True, exist_ok=True)
downloaded_snapshot = snapshot_download(
    repo_id=HF_DATASET_REPO,
    repo_type="dataset",
    local_dir=DATASET_ROOT,
    token=HF_TOKEN or None,
    allow_patterns=HF_ALLOW_PATTERNS,
)
print(f"Dataset snapshot: {downloaded_snapshot}")
print("Top-level entries:", sorted(path.name for path in DATASET_ROOT.iterdir()))

# **8. Prepare the Canonical `frames.parquet`**

If the snapshot already includes `data/metadata/frames.parquet`, the notebook leaves it unchanged. Otherwise, `scripts/prepare_data.py` joins the official mapping with the keyframe images and preserves the exact `frame_id → video_id → frame_idx` mapping. The cell fails immediately when the dataset layout is incomplete or mapping rows do not match images, preventing invalid metadata from being generated silently.

In [ ]:
runtime_env = os.environ.copy()
runtime_env["PYTHONPATH"] = str(PROJECT_DIR / "src")

if FRAMES_PATH.is_file():
    print(f"Reuse existing frames file: {FRAMES_PATH}")
else:
    FRAMES_PATH.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        [
            str(PYTHON),
            "scripts/prepare_data.py",
            "--dataset-root", str(DATASET_ROOT),
            "--output", str(FRAMES_PATH),
        ],
        cwd=PROJECT_DIR,
        env=runtime_env,
        check=True,
    )

# **9. Validate Inputs Before Encoding**

This cell validates the minimum schema, row count, and a sample of image paths. It provides a fast way to detect a misplaced `frames.parquet` or image paths that no longer resolve against the dataset root before loading the model and starting a long encoding run.

In [ ]:
frames = pd.read_parquet(FRAMES_PATH)
required_columns = {"frame_id", "video_id", "frame_idx", "image_path"}
missing_columns = required_columns.difference(frames.columns)
if missing_columns:
    raise ValueError(f"frames.parquet is missing columns: {sorted(missing_columns)}")

sample_paths = [DATASET_ROOT / value for value in frames["image_path"].head(10)]
missing_samples = [str(path) for path in sample_paths if not path.is_file()]
if missing_samples:
    raise FileNotFoundError(f"Sample images were not found: {missing_samples[:3]}")

print(f"Frames: {len(frames):,}")
print(f"Videos: {frames['video_id'].nunique():,}")
display(frames.head(3))

# **10. Run `build_embeddings.py`**

The script encodes every image with the configured model, writes normalized embeddings and their exact frame mapping to `artifacts/embeddings`, and then builds the FAISS index in `artifacts/indexes`. Subprocess output is streamed directly into the notebook.

The first run must download the model checkpoint. On CPU, both the first batch and the complete corpus can take a long time. The `Embedding dimension: pending` log message means the model has loaded and the pipeline is starting its first encoding batch. If RAM or VRAM is insufficient, reduce the batch size in section 5, rerun section 6, and then rerun this cell.

In [ ]:
build_command = [
    str(PYTHON),
    "scripts/build_embeddings.py",
    "--config", str(RUNTIME_CONFIG),
    "--dataset-root", str(DATASET_ROOT),
    "--frames", str(FRAMES_PATH),
    "--output", str(OUTPUT_ROOT),
    "--log-level", "INFO",
]

process = subprocess.Popen(
    build_command,
    cwd=PROJECT_DIR,
    env=runtime_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert process.stdout is not None
for output_line in process.stdout:
    print(output_line, end="")

return_code = process.wait()
if return_code != 0:
    raise subprocess.CalledProcessError(return_code, build_command)
print("Embedding build completed.")

# **11. Validate Local Artifacts**

A successful run must create three files in `artifacts/embeddings`: the `.npy` vectors, the exact frame mapping `.parquet`, and the `.yaml` metadata. This cell verifies that all three files exist and reports their sizes before upload. The FAISS index is not uploaded here because this notebook intentionally uploads only `artifacts/embeddings`.

In [ ]:
expected_artifacts = [
    EMBEDDINGS_DIR / "visual_embeddings.npy",
    EMBEDDINGS_DIR / "frame_mapping.parquet",
    EMBEDDINGS_DIR / "metadata.yaml",
]
missing_artifacts = [str(path) for path in expected_artifacts if not path.is_file()]
if missing_artifacts:
    raise FileNotFoundError(f"Missing embedding artifacts: {missing_artifacts}")

for artifact in expected_artifacts:
    size_mib = artifact.stat().st_size / (1024 ** 2)
    print(f"{artifact.relative_to(PROJECT_DIR)}: {size_mib:,.2f} MiB")

# **12. Upload `artifacts/embeddings` to Hugging Face**

This cell uploads only the embedding directory to the target dataset repository. Hugging Face Hub handles large files through its upload/LFS mechanism. If the target repository does not belong to your account, the token must have write access to it. Change `HF_UPLOAD_REPO` to your own dataset repository if you cannot write to `huylhn1810/aic_2025`.

In [ ]:
if not HF_TOKEN.strip():
    raise ValueError("Set HF_TOKEN in the configuration cell before uploading.")

hf_api = HfApi(token=HF_TOKEN)
commit_info = hf_api.upload_folder(
    repo_id=HF_UPLOAD_REPO,
    repo_type="dataset",
    folder_path=str(EMBEDDINGS_DIR),
    path_in_repo=HF_UPLOAD_PATH,
    commit_message=f"Upload {MODEL_NAME} visual embeddings",
)
print(f"Upload complete: {commit_info.commit_url}")

# **13. Verify the Uploaded Files on Hugging Face**

The final verification reads the current repository file list and filters it by `artifacts/embeddings`. The result should contain the `.npy` vectors, Parquet mapping, and YAML metadata. Another VM can then call `snapshot_download` with `allow_patterns=["artifacts/embeddings/**"]` to download only these artifacts.

In [ ]:
remote_files = hf_api.list_repo_files(
    repo_id=HF_UPLOAD_REPO,
    repo_type="dataset",
)
uploaded_files = [path for path in remote_files if path.startswith(f"{HF_UPLOAD_PATH}/")]
if not uploaded_files:
    raise RuntimeError(f"No files were found under {HF_UPLOAD_PATH} on the Hub.")

print("Remote embedding artifacts:")
for remote_path in uploaded_files:
    print(f"- {remote_path}")

# **14. Completion and Operational Notes**

- Do not share or commit the notebook after adding a token. Remove the token value and related cell output before saving it.
- The download step is safe to rerun; Hugging Face fetches only missing or changed files.
- Do not run two builds concurrently against the same `artifacts/` directory because both processes write to the same outputs.
- When changing the model or dataset version, upload to a different path or repository to avoid mixing incompatible artifacts.
- On another VM, download only `artifacts/embeddings/**` when rebuilding the index. Re-encoding still requires the keyframes and `frames.parquet`.